# 01 · Sumber data dan identitas molekul

**File:** `01_Sumber_Data.ipynb`

Tujuan: melihat 15 tabel dari lima sumber yang benar-benar dibaca, lalu mengikuti pencocokan Stimulus/CID/CAS ke SMILES.

**Urutan belajar:** README.md → notebook ini → 02_Preprocessing_dan_Audit_Label.ipynb.

**Mode:** penelusuran implementasi dan pemeriksaan data di memori. Run All tidak melatih model, tidak mengunduh data, dan tidak menimpa hasil penelitian. Setiap notebook membaca inputnya sendiri; tidak memerlukan variabel dari notebook lain.

Kode ML hanya memiliki satu sumber di `KODE_ML/research/`. Sel tampilan source membaca fungsi yang benar-benar diimpor; output kode itu bukan implementasi kedua. Sel berikutnya memanggil fungsi tersebut. Data/model besar tetap di folder artefak proyek.

## Persiapan
Cari folder KODE_ML dari lokasi notebook. Import mengambil kode lokal dan menampilkan lokasi aktual supaya asal kode dapat diperiksa.

In [1]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
cari = [Path.cwd(), *Path.cwd().parents]
KODE = next((p if (p / "research/runtime.py").is_file() and p.name == "KODE_ML"
             else p / "KODE_ML" for p in cari
             if (p / "research/runtime.py").is_file() and p.name == "KODE_ML"
             or (p / "KODE_ML/research/runtime.py").is_file()), None)
if KODE is None:
    raise FileNotFoundError("Buka notebook dari folder proyek atau KODE_ML.")
sys.path.insert(0, str(KODE))
from research.runtime import ROOT, read, digest
from belajar import tampilkan_sumber, tampilkan_awal
import pandas as pd
import numpy as np
from IPython.display import display, Markdown
pd.set_option("display.max_colwidth", 85)
print("Kode aktif:", KODE)


Kode aktif: C:\Users\ACER\Documents\Marvel\Skripsi\Project Skripsi\Perfume-MultiLabel-Classifier\KODE_ML


## Konfigurasi sumber

Input: konfigurasi sumber lokal. Output: lima sumber, commit yang dikunci, dan aturan label. Angka 109 belum dipakai untuk membuat daftar istilah.

In [2]:
from research import preprocessing as prep, source_identity as identity
config = prep.load_config()
display(pd.DataFrame(config['sources']).T)
display(config['labels'])
print('Commit snapshot:', config['revision'])

,molecules,stimuli,behavior
goodscents,goodscents/molecules.csv,goodscents/stimuli.csv,goodscents/behavior.csv
ifra,ifra_2019/molecules.csv,ifra_2019/stimuli.csv,ifra_2019/behavior.csv
leffingwell,leffingwell/molecules.csv,leffingwell/stimuli.csv,leffingwell/behavior.csv
arctander,arctander_1960/molecules.csv,arctander_1960/stimuli.csv,arctander_1960/behavior_1_sparse.csv
sigma,sigma_2014/molecules.csv,sigma_2014/stimuli.csv,sigma_2014/behavior.csv


{'taxonomy': 'IFRA 2019 observed descriptors',
 'min_train_positive': 30,
 'max_labels': None}

Commit snapshot: 8054ea98ed675005ec10e67359902f500e4911b0


## load_config

Memeriksa bahwa tepat lima sumber dipilih, commit ditulis lengkap, tidak ada batas top-N label, dan kebijakan grup sesuai. Ini validasi konfigurasi, bukan pemilihan hasil terbaik.

In [3]:
tampilkan_sumber(prep.load_config)

**Source aktual:** `KODE_ML/research/preprocessing.py`, baris 30–40. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `91aaeca7a9a9b6bb0cfae5349168c1205f62fb588496961c58193be96fb7ef14`.

```python
def load_config(path=CONFIG):
    config = json.loads(Path(path).read_text(encoding="utf-8"))
    if set(config["sources"]) != REQUIRED_SOURCES:
        raise ValueError("Exactly GoodScents, IFRA 2019, Leffingwell, Arctander 1960 and Sigma 2014 are required.")
    if not re.fullmatch(r"[0-9a-f]{40}", config["revision"]):
        raise ValueError("Use a full pinned commit SHA.")
    if config["labels"]["max_labels"] is not None:
        raise ValueError("This protocol does not impose a top-N label cap.")
    if config["split"].get("group_policy") != POLICY:
        raise ValueError("The active pipeline requires structure/feature grouped splitting.")
    return config

```

## load_sources

Membaca CSV lokal sebagai string agar CID/CAS tidak berubah menjadi angka pecahan. Setiap file diverifikasi SHA-256. Tidak ada unduhan saat notebook dibuka.

In [4]:
tampilkan_sumber(prep.load_sources)

**Source aktual:** `KODE_ML/research/preprocessing.py`, baris 42–57. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `91aaeca7a9a9b6bb0cfae5349168c1205f62fb588496961c58193be96fb7ef14`.

```python
def load_sources(config):
    """Baca snapshot yang sudah ada. Tidak mengunduh atau mengubah cache."""
    cache = ROOT / "data/snapshots/perfume-five-v1" / config["revision"]
    manifest = read(cache / "manifest.json")
    tables, selected = {}, {}
    for source, files in config["sources"].items():
        for kind, relative in files.items():
            path = cache / relative
            if relative not in manifest or digest(path) != manifest[relative]["sha256"]:
                raise ValueError(f"Source snapshot changed: {relative}")
            frame = pd.read_csv(path, dtype=str)
            if frame.empty:
                raise ValueError(f"Empty table: {relative}")
            tables[f"{source}_{kind}"] = frame
            selected[relative] = manifest[relative]
    return tables, selected

```

## Baca snapshot dan ukur tabel

Input: config. Output: tabel data mentah dan manifest. Baris molecules, stimuli, dan behavior punya unit berbeda; jangan dijumlahkan sebagai jumlah molekul training.

In [5]:
tables, source_manifest = prep.load_sources(config)
inventory = pd.DataFrame([{'tabel': key, 'baris': len(frame), 'kolom': len(frame.columns),
                          'nilai_kosong': int(frame.isna().sum().sum())} for key,frame in tables.items()])
display(inventory)
for key in ('goodscents_behavior','ifra_behavior','sigma_behavior'):
    print(key, list(tables[key].columns)[:12])
    display(tables[key].iloc[:3,:7])

,tabel,baris,kolom,nilai_kosong
0,goodscents_molecules,4565,5,6
1,goodscents_stimuli,4626,5,3265
2,goodscents_behavior,4626,2,4
3,ifra_molecules,1060,5,0
4,ifra_stimuli,1060,2,0
5,ifra_behavior,1146,4,0
6,leffingwell_molecules,3522,5,36
7,leffingwell_stimuli,3522,7,190
8,leffingwell_behavior,3522,114,0
9,arctander_molecules,2751,5,3


goodscents_behavior ['Stimulus', 'Descriptors']


,Stimulus,Descriptors
0,100-06-1,sweet;vanilla;cherry maraschino cherry;powdery;anisic;balsamic;hawthorn;acacia
1,100-09-4,phenolic;animal;fecal;medicinal
2,100-42-5,sweet;plastic;floral;balsamic


ifra_behavior ['Stimulus', 'Descriptor 1', 'Descriptor 2', 'Descriptor 3']


,Stimulus,Descriptor 1,Descriptor 2,Descriptor 3
0,439711,Earthy,Woody,Camphoraceous
1,16213731,Woody,Balsamic,Nutty
2,227829,Woody,Smoky,Rose


sigma_behavior ['Stimulus', 'descriptors', 'potato', 'marigold', 'anise', 'herba-', 'caraway', 'tropical', 'rose', 'brandy', 'coffee', 'creamy']


,Stimulus,descriptors,potato,marigold,anise,herba-,caraway
0,-69,['spicy'],False,False,False,False,False
1,-68,['cinnamon'],False,False,False,False,False
2,-67,"['nutty', 'ethereal', 'sweet']",False,False,False,False,False


## Library dan aturan identitas

Pandas melakukan join tabel; RDKit mengecek apakah SMILES valid dan terdiri dari satu komponen. Kode berikut berasal dari file lokal yang sama dengan yang dipakai cleaning.

In [6]:
tampilkan_awal(identity)

**Import dan konstanta aktual:** `KODE_ML/research/source_identity.py`. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
"""Pencocokan identitas molekul dan perapian istilah; tanpa konfigurasi lama."""
from collections import defaultdict
import re
import pandas as pd
from rdkit import Chem
_NON_ALNUM = re.compile(r"[^a-z0-9 ]+")
_WS = re.compile(r"\s+")

```

## MoleculeError

Menambahkan kode kesalahan agar invalid SMILES dan campuran bisa dibedakan.

In [7]:
tampilkan_sumber(identity.MoleculeError)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 9–12. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
class MoleculeError(ValueError):
    def __init__(self, code, message):
        super().__init__(message)
        self.code = code

```

## parse_molecule

Input satu string SMILES; output objek molekul RDKit. SMILES tidak valid atau beberapa komponen ditolak, tidak ditebak.

In [8]:
tampilkan_sumber(identity.parse_molecule)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 15–23. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def parse_molecule(smiles):
    if not isinstance(smiles, str) or not smiles.strip():
        raise MoleculeError("INVALID_SMILES", "Enter a non-empty SMILES string.")
    mol = Chem.MolFromSmiles(smiles.strip())
    if mol is None or mol.GetNumAtoms() == 0:
        raise MoleculeError("INVALID_SMILES", "The SMILES string cannot be parsed.")
    if len(Chem.GetMolFrags(mol)) != 1:
        raise MoleculeError("UNSUPPORTED_MIXTURE", "Prediction supports one connected molecule. Mixture odor has not been validated.")
    return mol

```

## canonical_smiles

Mengubah struktur valid ke penulisan baku. Stereo dipertahankan pada identitas molekul; hasil None menandai identitas gagal.

In [9]:
tampilkan_sumber(identity.canonical_smiles)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 26–30. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def canonical_smiles(smiles):
    try:
        return Chem.MolToSmiles(parse_molecule(smiles), isomericSmiles=True)
    except MoleculeError:
        return None

```

## _columns

Membuat kolom pengenal tetap dapat dibaca ketika sumber menyimpan pengenal sebagai indeks.

In [10]:
tampilkan_sumber(identity._columns)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 38–41. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _columns(frame):
    if frame.index.name and frame.index.name not in frame:
        frame = frame.reset_index()
    return frame.copy()

```

## _key

Menyamakan format kunci join: spasi dan akhiran .0. Ini hanya ID, bukan mengubah struktur kimia.

In [11]:
tampilkan_sumber(identity._key)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 44–45. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _key(series):
    return series.astype("string").str.strip().str.replace(r"\.0$", "", regex=True).replace("", pd.NA)

```

## _unique

Mengharuskan satu kunci punya satu nilai identitas. Konflik menghasilkan error agar join tidak menggandakan data diam-diam.

In [12]:
tampilkan_sumber(identity._unique)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 48–54. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _unique(frame, key, value):
    frame = frame[[key, value]].dropna().copy()
    frame[key] = _key(frame[key])
    frame = frame.dropna().drop_duplicates()
    if frame[key].duplicated().any():
        raise ValueError(f"Conflicting {key} -> {value} mapping")
    return frame

```

## _attach_smiles

Behavior digabung kiri melalui Stimulus→CID→SMILES. validate=many_to_one mencegah satu baris perilaku berubah menjadi banyak hasil join. Identitas kosong tetap ditandai kosong.

In [13]:
tampilkan_sumber(identity._attach_smiles)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 57–71. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _attach_smiles(beh, sti, mol):
    beh, sti, mol = map(_columns, (beh, sti, mol))
    cid = "CID" if "CID" in sti else "new_CID"
    if cid not in sti or "Stimulus" not in beh or "Stimulus" not in sti:
        raise ValueError("Explicit Stimulus -> CID bridge is required")
    smiles_column = next((c for c in ("IsomericSMILES", "SMILES", "CanonicalSMILES") if c in mol), None)
    if smiles_column is None or "CID" not in mol:
        raise ValueError("Explicit CID -> SMILES mapping is required")
    beh["Stimulus"], sti["Stimulus"] = _key(beh["Stimulus"]), _key(sti["Stimulus"])
    sti[cid], mol["CID"] = _key(sti[cid]), _key(mol["CID"])
    bridge = _unique(sti, "Stimulus", cid).rename(columns={cid: "__cid__"})
    lookup = _unique(mol, "CID", smiles_column).rename(columns={"CID": "__cid__", smiles_column: "__smiles__"})
    # Null identifiers are excluded from mappings; pandas must not match null keys.
    frame = beh.merge(bridge, on="Stimulus", how="left", validate="many_to_one")
    return frame.merge(lookup, on="__cid__", how="left", validate="many_to_one")

```

## _cas_bridge

Mengumpulkan pasangan CAS dan struktur dari sumber lain. CAS dengan beberapa identitas berbeda dianggap ambigu dan tidak dipakai untuk menjawab secara paksa.

In [14]:
tampilkan_sumber(identity._cas_bridge)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 95–112. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _cas_bridge(dfs, frames):
    identities = defaultdict(set)
    for source, frame in frames.items():
        sti = _columns(dfs[f"{source}_stimuli"])
        cas_col = next((c for c in sti if c.lower() == "cas"), None)
        if source == "goodscents":
            pairs = frame[["Stimulus", "__smiles__"]].rename(columns={"Stimulus": "__cas__"})
        elif cas_col:
            sti["Stimulus"] = _key(sti["Stimulus"])
            mapping = _unique(sti, "Stimulus", cas_col).rename(columns={cas_col: "__cas__"})
            pairs = frame[["Stimulus", "__smiles__"]].merge(mapping, on="Stimulus", validate="many_to_one")
        else:
            continue
        for cas, smiles in pairs[["__cas__", "__smiles__"]].itertuples(index=False, name=None):
            canonical = canonical_smiles(smiles)
            if isinstance(cas, str) and canonical:
                identities[cas.strip()].add(canonical)
    return {k: next(iter(v)) for k, v in identities.items() if len(v) == 1}, sorted(k for k,v in identities.items() if len(v)>1)

```

## _attach_sigma

Sigma memerlukan seluruh alias CAS mengarah ke satu identitas yang jelas. Bila satu alias gagal atau bertentangan, identitas ditandai tidak terselesaikan.

In [15]:
tampilkan_sumber(identity._attach_sigma)

**Source aktual:** `KODE_ML/research/source_identity.py`, baris 78–92. Kode di bawah dibaca dari objek yang diimpor. SHA-256: `36fb057eb35996f5a0f386712d6ce07896cb4ff9ef3675ead78325e8fecbccc6`.

```python
def _attach_sigma(beh, sti, cas_map):
    """Require every listed CAS alias to resolve to one validated identity."""
    beh, sti = map(_columns, (beh, sti))
    beh["Stimulus"], sti["Stimulus"] = _key(beh["Stimulus"]), _key(sti["Stimulus"])
    mapping, excluded = [], []
    for stimulus, rows in sti.dropna(subset=["Stimulus"]).groupby("Stimulus"):
        aliases = rows["CAS"].dropna().astype(str).str.strip().unique().tolist()
        identities = {cas_map.get(cas) for cas in aliases}
        if aliases and None not in identities and len(identities) == 1:
            mapping.append({"Stimulus": stimulus, "__smiles__": next(iter(identities))})
        else:
            excluded.append({"stimulus": stimulus, "cas": aliases,
                             "reason": "unresolved_or_conflicting_cas_aliases"})
    bridge = pd.DataFrame(mapping, columns=["Stimulus", "__smiles__"])
    return beh.merge(bridge, on="Stimulus", how="left", validate="many_to_one"), excluded

```

## Ikuti contoh join GoodScents

Input: tiga tabel mentah GoodScents. Output: jumlah baris tetap serta tambahan __cid__ dan __smiles__. Missing identity belum dibuang pada tahap ini.

In [16]:
attached = identity._attach_smiles(tables['goodscents_behavior'], tables['goodscents_stimuli'], tables['goodscents_molecules'])
assert len(attached) == len(tables['goodscents_behavior'])
display(attached[['Stimulus','__cid__','__smiles__']].head())
print('Identitas kosong:', int(attached['__smiles__'].isna().sum()))

,Stimulus,__cid__,__smiles__
0,100-06-1,7476,CC(=O)C1=CC=C(C=C1)OC
1,100-09-4,7478,COC1=CC=C(C=C1)C(=O)O
2,100-42-5,7501,C=CC1=CC=CC=C1
3,100-51-6,244,C1=CC=C(C=C1)CO
4,100-52-7,240,C1=CC=C(C=C1)C=O


Identitas kosong: 0


Satu baris behavior belum tentu satu molekul unik lintas sumber. Penggabungan molekul yang sama dilakukan sesudah identitas dan label lolos pemeriksaan.

## Cek pemahaman
- Kenapa label tidak langsung digabung berdasarkan nama bahan? Karena nama bisa ambigu; pipeline menelusuri identitas molekul.
- Apakah lima sumber hanya berisi parfum? Tidak. Ini sumber yang dipilih untuk konteks penelitian aroma; seluruh isinya tidak otomatis eksklusif parfum.

Lanjut ke notebook 02 untuk melihat aturan pembersihan dan audit label.